In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

# 1. LOAD AND PREPROCESS DATA
# Load the FAOSTAT Kenyan Agricultural Production dataset
df_raw = pd.read_csv('Kenyas_Agricultural_Production.xlsx - FAOSTAT_data_en_3-26-2023.csv')


In [2]:
# We want to identify "Significant Production" to create a meaningful association.
# We will define this as any crop production that is above the median value for that specific year.
year_medians = df_raw.groupby('Year')['Value'].transform('median')
df_significant = df_raw[df_raw['Value'] >= year_medians]

# Group by Year to create "Baskets" of crops produced together
transactions = df_significant.groupby('Year')['Item'].apply(list).values.tolist()

# 2. TRANSFORM DATA FOR ARM
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_basket = pd.DataFrame(te_ary, columns=te.columns_)

# 3. IMPLEMENT ALGORITHM 1: APRIORI
# Setting min_support to 0.4 (crops that appear together in 40% of the years)
print("--- Running Apriori Algorithm (Crop Co-occurrence) ---")
frequent_itemsets_apriori = apriori(df_basket, min_support=0.4, use_colnames=True)
rules_apriori = association_rules(frequent_itemsets_apriori, metric="lift", min_threshold=1.1)

# Sort and display top rules
top_rules_apriori = rules_apriori[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False)
print(top_rules_apriori.head(5))

# 4. IMPLEMENT ALGORITHM 2: FP-GROWTH
print("\n--- Running FP-Growth Algorithm (Crop Co-occurrence) ---")
frequent_itemsets_fp = fpgrowth(df_basket, min_support=0.4, use_colnames=True)
rules_fp = association_rules(frequent_itemsets_fp, metric="lift", min_threshold=1.1)

# Sort and display top rules
top_rules_fp = rules_fp[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False)
print(top_rules_fp.head(5))

# 5. SUMMARY
print(f"\nAnalysis complete. Apriori generated {len(rules_apriori)} rules; FP-Growth generated {len(rules_fp)} rules.")

NameError: name 'df_raw' is not defined